#### Creating target table

In [0]:
from delta.tables import DeltaTable
DeltaTable.create(spark)\
    .tableName("employee_demo")\
    .addColumn("emp_id", "INT")\
    .addColumn("emp_name", "STRING")\
    .addColumn("gender","STRING")\
    .addColumn("salary","INT")\
    .addColumn("dept", "STRING")\
    .property("desciption","table create for demo purpose")\
    .location("/FileStore/tables/delta/path_employee_demo")\
    .execute()

In [0]:
%sql
SELECT * FROM employee_demo;

In [0]:
%sql
INSERT INTO employee_demo VALUES (100, "Stephen", "M", 2000, "IT")

In [0]:
%sql
SELECT * FROM employee_demo;

In [0]:
%sql
DESC EXTENDED employee_demo;

#### creating source table

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

employee_data = [
    (200, "Philipp", "M", 8000, "HR", "test data")
]

employee_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("emp_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("dept", StringType(), True),
    StructField("additionalcolumn1", StringType(), True)
])

df = spark.createDataFrame(data=employee_data, schema=employee_schema)

df.show()
df.printSchema()


In [0]:
# since source table and target tables have mismatched column, so we get schema mismatch error.
df.write.format("delta").mode("append").saveAsTable("employee_demo")

#### Enable schema evolution with mergeSchema - "true"

In [0]:
# using mergeSchema mode, we allowed schema evolution
df.write.option("mergeSchema", "True").format("delta").mode("append").saveAsTable("employee_demo")

In [0]:
%sql
SELECT * FROM employee_demo;

In [0]:
# deleting old column "additionalcolumn1" and adding new column "additionalcolumn2"
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

employee_data = [
    (300, "David", "M", 8000, "HR", "dummy data")
]

employee_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("emp_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("dept", StringType(), True),
    StructField("additionalcolumn2", StringType(), True)
])

df = spark.createDataFrame(data=employee_data, schema=employee_schema)

df.show()
df.printSchema()


In [0]:
df.write.option("mergeSchema","true").format("delta").mode("append").saveAsTable("employee_demo")

In [0]:
%sql
SELECT * FROM employee_demo;

#### Enbale schema evolution with autoMerge